In [45]:
import pandas as pd
import numpy as np
import glob
import os
import re
from sklearn.model_selection import train_test_split

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns


In [46]:
dataset_path = "../../../DATASET_SCORE_TEST_EOI+2026/**/*.csv"
all_files = glob.glob(dataset_path, recursive=True)
# Filter: exclude files dengan nama diawali "MASTER_DATA_EOI"
pattern = re.compile(r"^MASTER_DATA_EOI")

filtered_files = [
    file for file in all_files if not pattern.match(os.path.basename(file))
]

df_list = []
for file in filtered_files:
    df_temp = pd.read_csv(file)
    df_list.append(df_temp)

# Gabungkan seluruh data
df_raw = pd.concat(df_list, ignore_index=True)

print(f"Total file ditemukan     : {len(all_files)}")
print(f"Total baris data awal    : {df_raw.shape[0]}")
df_raw.head()


Total file ditemukan     : 72
Total baris data awal    : 203141


,As At Month,Visa Type,Occupation,EOI Status,Count EOIs,English Test Score,State
0,2026-01-31,189PTS Points-Tested Stream,132211 Finance Manager,CLOSED,<20,0,ACT
1,2026-01-31,189PTS Points-Tested Stream,132511 Research and Development Manager,CLOSED,<20,0,ACT
2,2026-01-31,189PTS Points-Tested Stream,133111 Construction Project Manager,SUBMITTED,2100,0,ACT
3,2026-01-31,189PTS Points-Tested Stream,133111 Construction Project Manager,LODGED,223,0,ACT
4,2026-01-31,189PTS Points-Tested Stream,133111 Construction Project Manager,INVITED,<20,0,ACT


In [47]:
df_raw["Count EOIs"] = df_raw["Count EOIs"].replace("<20", 10)


In [48]:
df_raw["Count EOIs"] = df_raw["Count EOIs"].astype(int)


In [49]:
df_raw["EOI Status"].value_counts()


EOI Status
SUBMITTED    62361
CLOSED       60366
LODGED       37730
HOLD         26564
INVITED      16120
Name: count, dtype: int64

In [50]:
feature_cols = [
    col for col in df_raw.columns if col not in ["As At Month", "EOI Status"]
]

# Temukan duplikat berdasarkan fitur saja
duplicates = df_raw[df_raw.duplicated(subset=feature_cols, keep=False)]
print(f"Jumlah baris duplikat: {len(duplicates)}")
duplicates.sort_values(feature_cols)


Jumlah baris duplikat: 118734


,As At Month,Visa Type,Occupation,EOI Status,Count EOIs,English Test Score,State
8,2026-01-31,189PTS Points-Tested Stream,133211 Engineering Manager,LODGED,10,0,ACT
9,2026-01-31,189PTS Points-Tested Stream,133211 Engineering Manager,INVITED,10,0,ACT
10,2026-01-31,189PTS Points-Tested Stream,133211 Engineering Manager,HOLD,10,0,ACT
7818,2026-01-31,189PTS Points-Tested Stream,133211 Engineering Manager,LODGED,10,0,NSW
7819,2026-01-31,189PTS Points-Tested Stream,133211 Engineering Manager,INVITED,10,0,NSW
...,...,...,...,...,...,...,...
201266,2026-02-28,491SNR State or Territory Nominated - Regional,639211 Retail Buyer,CLOSED,10,20,WA
203139,2026-03-31,491SNR State or Territory Nominated - Regional,639211 Retail Buyer,SUBMITTED,10,20,WA
203140,2026-03-31,491SNR State or Territory Nominated - Regional,639211 Retail Buyer,CLOSED,10,20,WA
129825,2026-03-31,491SNR State or Territory Nominated - Regional,639211 Retail Buyer,SUBMITTED,23,10,VIC


In [50]:
key_cols = ["Occupation", "Visa Type", "English Test Score", "Count EOIs","State"]


# Kumpulkan keys per status
lodged_keys = set(
    df_raw[df_raw["EOI Status"] == "LODGED"][key_cols].apply(tuple, axis=1)
)
closed_keys = set(
    df_raw[df_raw["EOI Status"] == "CLOSED"][key_cols].apply(tuple, axis=1)
)
invited_keys = set(
    df_raw[df_raw["EOI Status"] == "INVITED"][key_cols].apply(tuple, axis=1)
)
hold_keys = set(df_raw[df_raw["EOI Status"] == "HOLD"][key_cols].apply(tuple, axis=1))
submitted_keys = set(
    df_raw[df_raw["EOI Status"] == "SUBMITTED"][key_cols].apply(tuple, axis=1)
)

# lodged_or_invited_keys = lodged_keys | invited_keys  # gabungan


# Baris SUBMITTED/INVITED yang kombinasinya sudah ada di LODGED
mask_drop = df_raw["EOI Status"].isin(["SUBMITTED", "INVITED","HOLD"]) & df_raw[
    key_cols
].apply(tuple, axis=1).isin(lodged_keys)

mask_closed = (df_raw["EOI Status"] == "CLOSED") & df_raw[key_cols].apply(
    tuple, axis=1
).isin(lodged_keys)

mask_to_drop = mask_drop | mask_closed

#HAPUS KOMBINASI BARIS YANG SUDAH ADA DI LODGED

df_clean = df_raw[~mask_to_drop].reset_index(drop=True)




print(f"Sebelum : {len(df_raw):,} baris")
print(f"Dihapus : {mask_to_drop.sum():,} baris (SUBMITTED,INVITED,HOLD DAN CLOSED yang sudah LODGED)")
print(f"Sesudah : {len(df_clean):,} baris")
print(f"\nDistribusi EOI Status setelah cleaning:")
print(df_clean["EOI Status"].value_counts())


Sebelum : 203,141 baris
Dihapus : 69,230 baris (SUBMITTED,INVITED,HOLD DAN CLOSED yang sudah LODGED)
Sesudah : 133,911 baris

Distribusi EOI Status setelah cleaning:
EOI Status
SUBMITTED    44335
CLOSED       40617
LODGED       37730
HOLD          7383
INVITED       3846
Name: count, dtype: int64


In [127]:
df_raw.to_csv('df_raw.csv',index=False)

In [51]:
idx_to_drop = df_clean[df_clean["EOI Status"].isin(['SUBMITTED','HOLD','INVITED'])].index
df_clean = df_clean.drop(index=idx_to_drop).reset_index(drop=True)


In [52]:
df_clean = df_clean[
    df_clean["Visa Type"] != "491FSR Family Sponsored - Regional"
]


In [53]:
df_clean["EOI Status"] = df_clean["EOI Status"].replace({'CLOSED':'NOT LODGED'})


In [54]:
df_clean.info()


<class 'pandas.DataFrame'>
Index: 74315 entries, 0 to 78346
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   As At Month         74315 non-null  str  
 1   Visa Type           74315 non-null  str  
 2   Occupation          74315 non-null  str  
 3   EOI Status          74315 non-null  str  
 4   Count EOIs          74315 non-null  int64
 5   English Test Score  74315 non-null  int64
 6   State               74315 non-null  str  
dtypes: int64(2), str(5)
memory usage: 4.5 MB


In [55]:
# ── Identifikasi 338 kombinasi ekstrem dari df_raw ───────────
combo_counts_raw = (
    df_clean.groupby(["Occupation", "English Test Score"])["EOI Status"]
    .agg(lodged=lambda x: (x == "LODGED").sum(), total="count")
    .reset_index()
)

combo_counts_raw["not_lodged"] = combo_counts_raw["total"] - combo_counts_raw["lodged"]
combo_counts_raw["type"] = combo_counts_raw.apply(
    lambda r: (
        "ALL NOT LODGED"
        if r["lodged"] == 0
        else ("ALL LODGED" if r["not_lodged"] == 0 else "MIXED")
    ),
    axis=1,
)

# ── Ambil hanya kombinasi MIXED ──────────────────────────────
mixed_combos = combo_counts_raw[combo_counts_raw["type"] == "MIXED"][
    ["Occupation", "English Test Score"]
]



In [56]:
# Cek kombinasi ekstrem yang dibuang
print("\nSample kombinasi ekstrem:")
print(
    combo_counts_raw[combo_counts_raw["type"] != "MIXED"][
        ["Occupation", "English Test Score", "lodged", "not_lodged", "total", "type"]
    ]
    .sort_values("total", ascending=False)
    .head(10)
    .to_string(index=False)
)



Sample kombinasi ekstrem:
                                        Occupation  English Test Score  lodged  not_lodged  total           type
             313211 Radiocommunications Technician                  10       0          60     60 ALL NOT LODGED
             313211 Radiocommunications Technician                  20       0          56     56 ALL NOT LODGED
             313211 Radiocommunications Technician                   0       0          56     56 ALL NOT LODGED
                     253512 Cardiothoracic Surgeon                  10       0          54     54 ALL NOT LODGED
                    211112 Dancer or Choreographer                  10       0          54     54 ALL NOT LODGED
                  134212 Nursing Clinical Director                  10       0          52     52 ALL NOT LODGED
                         254411 Nurse Practitioner                  10      48           0     48     ALL LODGED
254416 Registered Nurse (Developmental Disability)                   

In [ ]:
# ── Filter df ────────────────────────────────────────────
df_clean = df_clean.merge(
    mixed_combos, on=["Occupation", "English Test Score"], how="inner"
)

# ── Laporan ──────────────────────────────────────────────────
print("=" * 50)
print("  HASIL FILTER df_raw")
print("=" * 50)
print(f"  Sebelum : {len(df_raw):,} baris")
print(f"  Sesudah : {len(df_clean):,} baris")
print(f"  Dibuang : {len(df_raw) - len(df_clean):,} baris")
print(
    f"\n  Kombinasi ekstrem dibuang : {(combo_counts_raw['type'] != 'MIXED').sum():,}"
)
print(f"  Kombinasi MIXED tersisa   : {(combo_counts_raw['type'] == 'MIXED').sum():,}")
print(
    f"\n  LODGED rate sebelum : {(df_raw['EOI Status'] == 'LODGED').mean() * 100:.2f}%"
)
print(
    f"  LODGED rate sesudah : {(df_clean['EOI Status'] == 'LODGED').mean() * 100:.2f}%"
)




In [57]:
# Cek Occupation yang tidak pernah LODGED 
occ_stats = df_raw.groupby("Occupation")["EOI Status"]\
.agg( lodged=lambda x: (x == "LODGED").sum(),total="count").reset_index()

occ_stats["not_lodged"] = occ_stats["total"] - occ_stats["lodged"]
occ_stats["lodged_pct"] = occ_stats["lodged"] / occ_stats["total"] * 100

# Filter yang tidak pernah LODGED
never_lodged = occ_stats[occ_stats["lodged"] == 0].sort_values("total", ascending=False)

In [58]:
never_lodged


,Occupation,lodged,total,not_lodged,lodged_pct
370,313211 Radiocommunications Technician,0,283,283,0.0
67,211112 Dancer or Choreographer,0,270,270,0.0
76,212315 Program Director (Television or Radio),0,244,244,0.0
68,211212 Music Director,0,234,234,0.0
97,222213 Stockbroking Dealer,0,223,223,0.0
...,...,...,...,...,...
131,231114 Helicopter Pilot,0,27,27,0.0
262,253515 Otorhinolaryngologist,0,27,27,0.0
277,254212 Nurse Researcher,0,27,27,0.0
382,322312 Pressure Welder,0,26,26,0.0


In [22]:
never_lodged_list = never_lodged['Occupation'].tolist()

In [23]:
df_clean_filtered = df_clean[~df_clean["Occupation"].isin(never_lodged_list)]


In [58]:
print("\nSample kombinasi tersisa:")
print(
    combo_counts_raw[combo_counts_raw["type"] == "MIXED"][
        ["Occupation", "English Test Score", "lodged", "not_lodged", "total", "type"]
    ]
    .sort_values("total", ascending=False)
    .head(1000)
    .to_string(index=False)
)



Sample kombinasi tersisa:
                                                  Occupation  English Test Score  lodged  not_lodged  total  type
                              262112 ICT Security Specialist                  10      64          64    128 MIXED
                                 261111 ICT Business Analyst                  10      64          64    128 MIXED
                                 261111 ICT Business Analyst                  20      64          64    128 MIXED
                                    261313 Software Engineer                  10      64          64    128 MIXED
                                      261112 Systems Analyst                  10      64          64    128 MIXED
                                254499 Registered Nurses nec                  10      64          64    128 MIXED
                       312211 Civil Engineering Draftsperson                  10      64          64    128 MIXED
                                 221111 Accountant (General) 

In [55]:
from sklearn.preprocessing import LabelEncoder,StandardScaler


df_model = df_clean.copy()
le_occ = LabelEncoder()
Standardize = StandardScaler()
df_model = pd.get_dummies(df_model, columns=["Visa Type", "State"], dtype=int)
df_model["occupation_enc"] = le_occ.fit_transform(df_model["Occupation"])
df_model["Status_enc"]  = df_model['EOI Status'].map(
    {
        "LODGED": 1,  
        "NOT LODGED": 0
    }
)
drop_cols = ["Occupation", "As At Month", "EOI Status", "Status_enc"]
feature_cols = [c for c in df_model.columns if c not in drop_cols]


In [56]:
df_model['Occupation'].nunique()

484

In [57]:
df_model['EOI Status'].value_counts()

EOI Status
NOT LODGED    38097
LODGED        36218
Name: count, dtype: int64

In [60]:
feature_cols


['Count EOIs',
 'English Test Score',
 'Visa Type_189PTS Points-Tested Stream',
 'Visa Type_190SAS Skilled Australian Sponsored',
 'Visa Type_491SNR State or Territory Nominated - Regional',
 'State_ACT',
 'State_NSW',
 'State_NT',
 'State_QLD',
 'State_SA',
 'State_TAS',
 'State_VIC',
 'State_WA',
 'occupation_enc']

In [61]:
df_model[feature_cols].info()


<class 'pandas.DataFrame'>
Index: 74315 entries, 0 to 78346
Data columns (total 14 columns):
 #   Column                                                    Non-Null Count  Dtype
---  ------                                                    --------------  -----
 0   Count EOIs                                                74315 non-null  int64
 1   English Test Score                                        74315 non-null  int64
 2   Visa Type_189PTS Points-Tested Stream                     74315 non-null  int64
 3   Visa Type_190SAS Skilled Australian Sponsored             74315 non-null  int64
 4   Visa Type_491SNR State or Territory Nominated - Regional  74315 non-null  int64
 5   State_ACT                                                 74315 non-null  int64
 6   State_NSW                                                 74315 non-null  int64
 7   State_NT                                                  74315 non-null  int64
 8   State_QLD                                           

In [62]:
df_model['Occupation']

0                         132211 Finance Manager
1        132511 Research and Development Manager
2            133111 Construction Project Manager
3            133111 Construction Project Manager
4                     133211 Engineering Manager
                          ...                   
78342      452311 Diving Instructor (Open Water)
78343              511111 Contract Administrator
78344    511112 Program or Project Administrator
78345    511112 Program or Project Administrator
78346                        639211 Retail Buyer
Name: Occupation, Length: 74315, dtype: str

In [63]:
X = df_model[feature_cols]
y = df_model["Status_enc"]


In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [65]:
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    classification_report,
)


In [66]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


In [67]:
# SMOTE hanya di training set
rus = SMOTE(random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)
print(f"Train sebelum: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Train sesudah: {pd.Series(y_train_bal).value_counts().to_dict()}")
print(f"Test (tidak diubah): {pd.Series(y_test).value_counts().to_dict()}")


Train sebelum: {0: 30523, 1: 28929}
Train sesudah: {0: 30523, 1: 30523}
Test (tidak diubah): {0: 7574, 1: 7289}


In [68]:
from xgboost import XGBClassifier

# Handle imbalance
scale_pos_weight = (y_train_bal == 1).sum() / (y_train_bal == 0).sum()

# Train model
model = XGBClassifier(
    n_estimators=900,
    max_depth=9,
    learning_rate=0.06,
    subsample=1,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
)


In [69]:
model.fit(
    X_train_bal,
    y_train_bal
)


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [70]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.93      0.90      0.91      7574
           1       0.90      0.92      0.91      7289

    accuracy                           0.91     14863
   macro avg       0.91      0.91      0.91     14863
weighted avg       0.91      0.91      0.91     14863



In [71]:
from sklearn.metrics import f1_score

# Cek gap train vs test
train_f1 = f1_score(y_train, model.predict(X_train))
test_f1 = f1_score(y_test, model.predict(X_test))
print(f"Train F1 : {train_f1:.3f}")
print(f"Test  F1 : {test_f1:.3f}")
print(f"Gap      : {train_f1 - test_f1:.3f}")


Train F1 : 0.943
Test  F1 : 0.911
Gap      : 0.032


In [124]:
import joblib
joblib.dump(le_occ, "encoder_occupation.joblib")


['encoder_occupation.joblib']

In [42]:
# Tabel mapping lengkap
mapping = pd.DataFrame(
    {"occupation": le_occ.classes_, "encoded": range(len(le_occ.classes_))}
)

pd.set_option("display.max_rows", None)
print(mapping.to_string(index=False))
pd.reset_option("display.max_rows")


                                                  occupation  encoded
                                   121111 Aquaculture Farmer        0
                                        121211 Cotton Grower        1
                                        121212 Flower Grower        2
                                  121213 Fruit or Nut Grower        3
                     121214 Grain, Oilseed or Pasture Grower        4
                                         121215 Grape Grower        5
                                    121216 Mixed Crop Farmer        6
                                    121217 Sugar Cane Grower        7
                                     121221 Vegetable Grower        8
                                     121299 Crop Farmers nec        9
                                             121311 Apiarist       10
                                   121312 Beef Cattle Farmer       11
                                  121313 Dairy Cattle Farmer       12
                    

In [43]:
occ_dict = dict(zip(mapping["occupation"], mapping["encoded"]))

# ── Input ───────────────────────────────────────────────────
occupation_name = "233211 Civil Engineer"
English = 10
eoi_count = 499
state = "TAS"
visa_type = "190"  # "189", "190", atau "491"

# ── Mapping ─────────────────────────────────────────────────
STATE_COL_MAP = {
    "ACT": "State_ACT",
    "NSW": "State_NSW",
    "NT": "State_NT",
    "QLD": "State_QLD",
    "SA": "State_SA",
    "TAS": "State_TAS",
    "VIC": "State_VIC",
    "WA": "State_WA",
}

VISA_COL_MAP = {
    "189": "Visa Type_189SS Skilled Independent",  # ← sesuaikan nama kolom
    "190": "Visa Type_190SAS Skilled Australian Sponsored",
    "491": "Visa Type_491SNR State or Territory - Regional",
}

# ── Validasi input awal ──────────────────────────────────────
if state not in STATE_COL_MAP:
    raise ValueError(
        f"State '{state}' tidak valid! Pilih: {list(STATE_COL_MAP.keys())}"
    )

if visa_type not in VISA_COL_MAP:
    raise ValueError(
        f"Visa type '{visa_type}' tidak valid! Pilih: {list(VISA_COL_MAP.keys())}"
    )

# ── Encode occupation ────────────────────────────────────────
if occupation_name not in le_occ.classes_:
    raise ValueError(
        f"Occupation '{occupation_name}' tidak ditemukan di label encoder!"
    )

occ_enc = int(le_occ.transform([occupation_name])[0])

# ── Buat data dummy ──────────────────────────────────────────
data_dummy = pd.DataFrame(0, index=[0], columns=X_train.columns)
data_dummy["Count EOIs"] = eoi_count
data_dummy["occupation_enc"] = occ_enc
data_dummy['English Test Score'] = English

# Set State — FIX: sekarang pakai variabel state, bukan hardcode
state_col = STATE_COL_MAP[state]
if state_col in data_dummy.columns:
    data_dummy[state_col] = 1
else:
    print(f"⚠️  Kolom '{state_col}' tidak ada di training data!")

# Set Visa Type
visa_col = VISA_COL_MAP[visa_type]
if visa_col in data_dummy.columns:
    data_dummy[visa_col] = 1
else:
    print(f"⚠️  Kolom '{visa_col}' tidak ada di training data!")
    print(
        f"    Kolom visa yang tersedia: {[c for c in X_train.columns if 'Visa' in c]}"
    )

data_dummy = data_dummy.astype(float)

# ── Debug — cek kolom yang aktif ────────────────────────────
print("Kolom aktif (non-zero):")
print(data_dummy.loc[:, data_dummy.iloc[0] != 0].T.to_string())
print()

# ── Prediksi ─────────────────────────────────────────────────
prediksi_label = model.predict(data_dummy)[0]
prediksi_proba = model.predict_proba(data_dummy)[0]

status_map = {0: "NOT LODGED", 1: "LODGED"}
status = status_map.get(prediksi_label, str(prediksi_label))

# ── Output ───────────────────────────────────────────────────
print("=" * 55)
print("  HASIL PREDIKSI")
print("=" * 55)
print(f"  Occupation    : {occupation_name}")
print(f"  EOI Count     : {eoi_count}")
print(f"  State         : {state}")
print(f"  Visa Type     : {visa_type}")
print(f"  {'─' * 37}")
print(f"  Prediksi      : {status}")
print(f"  P(NOT LODGED) : {prediksi_proba[0] * 100:.2f}%")
print(f"  P(LODGED)     : {prediksi_proba[1] * 100:.2f}%")
print("=" * 55)


Kolom aktif (non-zero):
                                                   0
Count EOIs                                     499.0
English Test Score                              10.0
Visa Type_190SAS Skilled Australian Sponsored    1.0
State_TAS                                        1.0
occupation_enc                                 146.0

  HASIL PREDIKSI
  Occupation    : 233211 Civil Engineer
  EOI Count     : 499
  State         : TAS
  Visa Type     : 190
  ─────────────────────────────────────
  Prediksi      : NOT LODGED
  P(NOT LODGED) : 69.79%
  P(LODGED)     : 30.21%


In [50]:
# ── Cek balance per kombinasi occupation & english score ─────
print("=" * 60)
print("  CEK BALANCE PER KOMBINASI OCCUPATION & ENGLISH SCORE")
print("=" * 60)

# 1. Hitung distribusi per kombinasi
combo_stats = (
    df_model.groupby(["occupation_enc", "English Test Score"])["Status_enc"]
    .agg(lodged="sum", total="count")
    .reset_index()
)
combo_stats["not_lodged"] = combo_stats["total"] - combo_stats["lodged"]
combo_stats["lodged_pct"] = combo_stats["lodged"] / combo_stats["total"] * 100

print(f"\nTotal kombinasi unik : {len(combo_stats):,}")
print(
    f"Kombinasi yg HANYA NOT LODGED (lodged=0) : {(combo_stats['lodged'] == 0).sum():,}"
)
print(
    f"Kombinasi yg HANYA LODGED (not_lodged=0) : {(combo_stats['not_lodged'] == 0).sum():,}"
)
print(
    f"Kombinasi yg balance (40-60%) : {combo_stats['lodged_pct'].between(40, 60).sum():,}"
)

print(f"\nDistribusi LODGED rate per kombinasi:")
print(combo_stats["lodged_pct"].describe().round(2))


  CEK BALANCE PER KOMBINASI OCCUPATION & ENGLISH SCORE

Total kombinasi unik : 1,444
Kombinasi yg HANYA NOT LODGED (lodged=0) : 200
Kombinasi yg HANYA LODGED (not_lodged=0) : 138
Kombinasi yg balance (40-60%) : 809

Distribusi LODGED rate per kombinasi:
count    1444.00
mean       48.13
std        25.87
min         0.00
25%        41.99
50%        50.00
75%        58.14
max       100.00
Name: lodged_pct, dtype: float64


In [102]:
# Ganti sesuai Occupation & state yang dimaksud
occ_name = "231114 Helicopter Pilot"  # ← ganti
state_name = "WA"  # ← ganti

# ── Cek di data mentah ───────────────────────────────────────
print("=" * 60)
print(f"  CEK: {occ_name} | {state_name}")
print("=" * 60)

# 1. Overall Occupation
occ_data = df_clean[df_clean["Occupation"] == occ_name]
print(f"\n[1] Overall Occupation:")
print(f"    Total baris : {len(occ_data):,}")
print(f"    LODGED      : {(occ_data['EOI Status'] == 'LODGED').sum():,}")
print(f"    NOT LODGED  : {(occ_data['EOI Status'] == 'NOT LODGED').sum():,}")

# 2. Occupation di state tersebut
occ_state = df_clean[
    (df_clean["Occupation"] == occ_name) & (df_clean["State"] == state_name)
]
print(f"\n[2] OccuState {state_name}:")
print(f"    Total baris : {len(occ_state):,}")
print(f"    LODGED      : {(occ_state['EOI Status'] == 'LODGED').sum():,}")
print(f"    NOT LODGED  : {(occ_state['EOI Status'] == 'NOT LODGED').sum():,}")


# 4. Distribusi English Score
print(f"\n[4] Distribusi English Score di {state_name}:")
print(occ_state.groupby("EOI Status")["English Test Score"].value_counts())

# 5. Cek berapa baris total — kalau sedikit = overfitting!
print(f"\n[5] Sample data di {state_name}:")
print(
    occ_state[
        ["Occupation", "State",  "English Test Score", "EOI Status"]
    ]
    .head(10)
    .to_string(index=False)
)


  CEK: 231114 Helicopter Pilot | WA

[1] Overall Occupation:
    Total baris : 0
    LODGED      : 0
    NOT LODGED  : 0

[2] OccuState WA:
    Total baris : 0
    LODGED      : 0
    NOT LODGED  : 0

[4] Distribusi English Score di WA:
Series([], Name: count, dtype: int64)

[5] Sample data di WA:
Empty DataFrame
Columns: [Occupation, State, English Test Score, EOI Status]
Index: []


In [72]:
model.save_model("model_xgboost.json")
